In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def plot_split_violin(city_name, trains, bikes, config):
    print(f"  > Generating Split Violin Plot for {city_name}...")

    # 1. SETUP TIMELINE & SIGNALS
    # We need to process the whole timeline to get the pressure signal
    start_str, end_str = config['vis_start'], config['vis_end']
    zoom_start, zoom_end = pd.to_datetime(start_str), pd.to_datetime(end_str)

    timeline = pd.date_range(zoom_start, zoom_end, freq='1min')

    # Generate Pressure Signal (Theoretical Demand)
    # (Assuming you have ta.generate_signals available)
    import transport_analytics as ta
    df_trains = ta.standard_columns(trains)

    # Filter for speed
    t_mask = (df_trains['actual_arrival'] >= zoom_start) & (df_trains['actual_arrival'] <= zoom_end)
    subset_trains = df_trains[t_mask].copy()

    # Get the pressure signal (Series)
    sig_arr, _ = ta.generate_signals(subset_trains, timeline, config['params'])

    # 2. PREPARE BIKE DATA (Actual Rentals)
    # We just need the list of timestamps when rentals started
    b_mask = (bikes['start_time'] >= zoom_start) & (bikes['start_time'] <= zoom_end)
    rental_times = bikes.loc[b_mask, 'start_time']

    # 3. CONVERT TO "LONG FORMAT" FOR SEABORN
    # We need a DataFrame with columns: [Day, Hour, Type]

    data_rows = []

    # A. Bike Data (Discrete Events)
    # Simple: Just take every rental timestamp
    for t in rental_times:
        data_rows.append({
            'Day': t.strftime('%a'), # Mon, Tue...
            'DayNum': t.dayofweek,   # 0, 1... for sorting
            'Hour': t.hour + t.minute/60.0,
            'Type': 'Actual Rentals'
        })

    # B. Pressure Data (Continuous Signal)
    # Trick: Violin plots need "events". We simulate events by sampling the pressure curve.
    # If pressure is 100 at 08:00, we add the timestamp "08:00" 10 times (scaled down).

    # Resample to 10-min to reduce processing size, then sample
    sig_resampled = sig_arr.resample('10min').mean()
    sig_resampled = sig_resampled[sig_resampled > 0.1] # Ignore zeros

    # Normalization factor (don't create millions of rows)
    # We want roughly same number of "pressure events" as "bike events" for balanced plot
    target_count = len(rental_times)
    current_pressure_sum = sig_resampled.sum()
    scale_factor = target_count / current_pressure_sum if current_pressure_sum > 0 else 1

    for t, pressure_val in sig_resampled.items():
        # Number of "fake events" to generate proportional to pressure
        count = int(pressure_val * scale_factor)
        if count > 0:
            row = {
                'Day': t.strftime('%a'),
                'DayNum': t.dayofweek,
                'Hour': t.hour + t.minute/60.0,
                'Type': 'Passenger Pressure'
            }
            # Add this row 'count' times
            data_rows.extend([row] * count)

    # Create DataFrame
    df_viz = pd.DataFrame(data_rows)

    # Sort by Day of Week (Mon -> Sun)
    df_viz.sort_values('DayNum', inplace=True)

    # 4. PLOTTING
    plt.figure(figsize=(12, 8))
    sns.set_style("whitegrid")

    # The Split Violin
    ax = sns.violinplot(
        data=df_viz,
        x='Day',
        y='Hour',
        hue='Type',
        split=True,           # <--- The Magic Switch
        inner="quartile",     # Shows the median and quartiles inside
        palette={"Actual Rentals": "#004c6d", "Passenger Pressure": "#d55e00"}, # Blue vs Orange
        cut=0,                # Don't extend past data range
        scale='area'          # Normalize area so days are comparable
    )

    # Formatting
    plt.title(f"Synchronization Analysis: {city_name}\n(Left: Theoretical Demand | Right: Actual Usage)", fontsize=14, fontweight='bold')
    plt.ylabel("Time of Day", fontsize=12)
    plt.xlabel("")

    # Fix Y-axis to show 00:00 to 24:00 nicely
    ax.set_yticks(range(0, 25, 3))
    ax.set_yticklabels([f"{h:02d}:00" for h in range(0, 25, 3)])
    ax.set_ylim(4, 23) # Optional: Zoom in on active hours (e.g., 4 AM to 11 PM)

    # Add Rush Hour Lines (Optional Reference)
    plt.axhline(y=7, color='gray', linestyle='--', alpha=0.3)
    plt.axhline(y=16, color='gray', linestyle='--', alpha=0.3)

    # Move Legend
    plt.legend(title="", loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2, frameon=False)

    plt.tight_layout()
    plt.savefig(f"results/Violin_Sync_{city_name}.png", dpi=300)
    plt.show()

# --- EXAMPLE USAGE ---
# You can call this inside your main loop like this:
# plot_split_violin(city_key, df_trains, df_bikes, config)